# 03 — Hypothesis Testing & Policy Decision

## Objective

Formally evaluate whether the simulated new lending policy changes **default risk** and **approval rates**.

This notebook combines statistical rigor with banking/business interpretation.

### Primary outcome
Simulated default rate.

### Secondary outcome
Approval rate.

### Tests
- Two-proportion Z-test
- 95% confidence intervals
- Effect size
- Number Needed to Treat (NNT)
- Welch's t-tests for continuous baseline variables
- Chi-square tests for categorical variables

> **Important:** The experiment is simulated from historical Lending Club data. These tests evaluate our simulated experiment, not an actual Lending Club policy change.


## 1. Imports and load the experiment dataset


In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "credit_policy_experiment.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Run Notebook 01 first."
    )

df = pd.read_csv(DATA_PATH)

control = df[df["policy_group"] == "control"].copy()
treatment = df[df["policy_group"] == "treatment"].copy()

print(f"Total experiment population: {len(df):,}")
print(f"Control: {len(control):,}")
print(f"Treatment: {len(treatment):,}")


Total experiment population: 1,302,848
Control: 651,872
Treatment: 650,976


## 2. Define the hypotheses

### Primary hypothesis — default rate

**H₀:** The treatment policy does not change the default rate.

\[
H_0: p_T = p_C
\]

**H₁:** The treatment policy reduces the default rate.

\[
H_1: p_T < p_C
\]

We use a one-sided alternative because the simulated policy was explicitly designed to reduce default risk.

### Secondary hypothesis — approval rate

For approval, we use a two-sided test because a policy could theoretically increase or decrease approvals.

\[
H_0: p_T = p_C
\]

\[
H_1: p_T 
e p_C
\]


## 3. Primary outcome — default rates


In [6]:
control_default_count = int(control["simulated_default"].sum())
treatment_default_count = int(treatment["simulated_default"].sum())

control_n = len(control)
treatment_n = len(treatment)

control_default_rate = control_default_count / control_n
treatment_default_rate = treatment_default_count / treatment_n

absolute_reduction = control_default_rate - treatment_default_rate
relative_reduction = absolute_reduction / control_default_rate

default_summary = pd.DataFrame({
    "Group": ["Control", "Treatment"],
    "Borrowers": [control_n, treatment_n],
    "Defaults": [control_default_count, treatment_default_count],
    "Default Rate": [control_default_rate, treatment_default_rate]
})

display(default_summary)

print(f"Absolute reduction: {absolute_reduction:.4%}")
print(f"Relative reduction: {relative_reduction:.2%}")


,Group,Borrowers,Defaults,Default Rate
0,Control,651872,131036,0.201015
1,Treatment,650976,123099,0.189099


Absolute reduction: 1.1916%
Relative reduction: 5.93%


## 4. Two-proportion Z-test — default rate


In [7]:
# H0: p_treatment = p_control
# H1: p_treatment < p_control

counts = np.array([
    treatment_default_count,
    control_default_count
])

sample_sizes = np.array([
    treatment_n,
    control_n
])

z_stat, p_value = proportions_ztest(
    count=counts,
    nobs=sample_sizes,
    alternative="smaller"
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"One-sided p-value: {p_value:.6g}")


Z-statistic: -17.1622
One-sided p-value: 2.54787e-66


## 5. 95% confidence interval for the treatment effect


In [8]:
# Difference is defined as treatment - control.
# A negative value therefore represents a reduction in default.

p_t = treatment_default_rate
p_c = control_default_rate

se_diff = np.sqrt(
    (p_t * (1 - p_t) / treatment_n)
    + (p_c * (1 - p_c) / control_n)
)

difference = p_t - p_c

ci_low = difference - 1.96 * se_diff
ci_high = difference + 1.96 * se_diff

print(f"Treatment - Control difference: {difference:.4%}")
print(f"95% CI: [{ci_low:.4%}, {ci_high:.4%}]")

print(
    "\nBecause the difference is Treatment - Control, "
    "negative values indicate a reduction in default."
)


Treatment - Control difference: -1.1916%
95% CI: [-1.3276%, -1.0555%]

Because the difference is Treatment - Control, negative values indicate a reduction in default.


## 6. Effect size and Number Needed to Treat


In [9]:
# Absolute risk reduction
ARR = control_default_rate - treatment_default_rate

# Relative risk
relative_risk = treatment_default_rate / control_default_rate

# Relative risk reduction
relative_risk_reduction = 1 - relative_risk

# NNT
NNT = 1 / ARR if ARR > 0 else np.inf

# Cohen's h for two proportions
cohens_h = (
    2 * np.arcsin(np.sqrt(treatment_default_rate))
    - 2 * np.arcsin(np.sqrt(control_default_rate))
)

effect_summary = pd.DataFrame({
    "Metric": [
        "Absolute risk reduction",
        "Relative risk reduction",
        "Relative risk",
        "Cohen's h",
        "Number Needed to Treat"
    ],
    "Value": [
        ARR,
        relative_risk_reduction,
        relative_risk,
        cohens_h,
        NNT
    ]
})

display(effect_summary)

print(
    f"Approximately {NNT:.1f} borrowers need to receive "
    "the treatment policy to prevent one additional default, "
    "based on the simulated effect."
)


,Metric,Value
0,Absolute risk reduction,0.011916
1,Relative risk reduction,0.059278
2,Relative risk,0.940722
3,Cohen's h,-0.030075
4,Number Needed to Treat,83.922287


Approximately 83.9 borrowers need to receive the treatment policy to prevent one additional default, based on the simulated effect.


## 7. Business interpretation — default outcome


In [10]:
alpha = 0.05

if p_value < alpha:
    significance_statement = (
        "The simulated treatment produces a statistically significant "
        "reduction in default at the 5% level."
    )
else:
    significance_statement = (
        "The simulated treatment does not produce a statistically significant "
        "reduction in default at the 5% level."
    )

print(significance_statement)

print(
    f"\nThe observed reduction is {ARR:.2%} "
    f"({ARR * 100:.2f} percentage points)."
)

print(
    f"The relative reduction is {relative_risk_reduction:.2%}."
)

print(
    f"The estimated NNT is approximately {NNT:.0f} borrowers."
)

print(
    "\nImportant: statistical significance does not by itself establish "
    "that the policy is economically worthwhile. Expected loss and the "
    "approval tradeoff are evaluated later."
)


The simulated treatment produces a statistically significant reduction in default at the 5% level.

The observed reduction is 1.19% (1.19 percentage points).
The relative reduction is 5.93%.
The estimated NNT is approximately 84 borrowers.

Important: statistical significance does not by itself establish that the policy is economically worthwhile. Expected loss and the approval tradeoff are evaluated later.


## 8. Secondary outcome — approval rate


In [11]:
control_approval_count = int(control["approved"].sum())
treatment_approval_count = int(treatment["approved"].sum())

control_approval_rate = control_approval_count / control_n
treatment_approval_rate = treatment_approval_count / treatment_n

approval_difference = treatment_approval_rate - control_approval_rate

approval_summary = pd.DataFrame({
    "Group": ["Control", "Treatment"],
    "Applications": [control_n, treatment_n],
    "Approved": [control_approval_count, treatment_approval_count],
    "Approval Rate": [control_approval_rate, treatment_approval_rate]
})

display(approval_summary)

print(
    f"Treatment - Control approval difference: "
    f"{approval_difference:.4%}"
)


,Group,Applications,Approved,Approval Rate
0,Control,651872,651872,1.000000
1,Treatment,650976,635224,0.975802


Treatment - Control approval difference: -2.4198%


## 9. Two-proportion Z-test — approval rate


In [12]:
approval_counts = np.array([
    treatment_approval_count,
    control_approval_count
])

approval_sizes = np.array([
    treatment_n,
    control_n
])

approval_z, approval_p = proportions_ztest(
    count=approval_counts,
    nobs=approval_sizes,
    alternative="two-sided"
)

print(f"Z-statistic: {approval_z:.4f}")
print(f"Two-sided p-value: {approval_p:.6g}")

if approval_p < alpha:
    print(
        "The treatment approval rate is statistically different "
        "from the control approval rate."
    )
else:
    print(
        "No statistically significant approval-rate difference "
        "was detected."
    )


Z-statistic: -126.3595
Two-sided p-value: 0
The treatment approval rate is statistically different from the control approval rate.


## 10. Approximate number of lost approvals


In [13]:
approval_loss = control_approval_rate - treatment_approval_rate

estimated_lost_approvals = (
    treatment_n * approval_loss
)

print(
    f"Approval-rate reduction: {approval_loss:.2%}"
)

print(
    f"Approximate approvals lost per 100 treatment applicants: "
    f"{approval_loss * 100:.2f}"
)

print(
    f"Approximate approvals lost across the treatment group: "
    f"{estimated_lost_approvals:,.0f}"
)


Approval-rate reduction: 2.42%
Approximate approvals lost per 100 treatment applicants: 2.42
Approximate approvals lost across the treatment group: 15,752


## 11. Baseline balance — Welch's t-tests


In [14]:
continuous_features = [
    col for col in [
        "annual_income",
        "loan_amount",
        "dti_ratio",
        "employment_years",
        "interest_rate"
    ]
    if col in df.columns
]

balance_results = []

for feature in continuous_features:
    c = control[feature].dropna()
    t = treatment[feature].dropna()

    t_stat, p_val = stats.ttest_ind(
        t,
        c,
        equal_var=False
    )

    pooled_sd = np.sqrt(
        (c.var(ddof=1) + t.var(ddof=1)) / 2
    )

    smd = (
        (t.mean() - c.mean()) / pooled_sd
        if pooled_sd > 0 else np.nan
    )

    balance_results.append({
        "feature": feature,
        "control_mean": c.mean(),
        "treatment_mean": t.mean(),
        "difference": t.mean() - c.mean(),
        "t_statistic": t_stat,
        "p_value": p_val,
        "standardized_mean_difference": smd
    })

balance_results = pd.DataFrame(balance_results)

display(balance_results)


,feature,control_mean,treatment_mean,difference,t_statistic,p_value,standardized_mean_difference
0,annual_income,76129.269634,76272.801114,143.531480,1.169378,0.242252,0.002049
1,loan_amount,14406.149213,14422.289186,16.139972,1.059048,0.289578,0.001856
2,dti_ratio,18.168278,18.169410,0.001132,0.074713,0.940443,0.000131
3,employment_years,5.968037,5.971041,0.003003,0.451036,0.651963,0.000814
4,interest_rate,13.257704,13.254075,-0.003629,-0.435195,0.663421,-0.000763


## 12. Interpret baseline balance


In [15]:
balance_results["statistically_significant"] = (
    balance_results["p_value"] < alpha
)

balance_results["small_smd"] = (
    balance_results["standardized_mean_difference"].abs() < 0.10
)

display(
    balance_results[
        [
            "feature",
            "p_value",
            "statistically_significant",
            "standardized_mean_difference",
            "small_smd"
        ]
    ]
)

print(
    "\nInterpretation: With a very large sample, even tiny baseline "
    "differences can become statistically significant. Therefore, "
    "standardized mean differences are more informative for judging "
    "whether randomization created practically comparable groups."
)


,feature,p_value,statistically_significant,standardized_mean_difference,small_smd
0,annual_income,0.242252,False,0.002049,True
1,loan_amount,0.289578,False,0.001856,True
2,dti_ratio,0.940443,False,0.000131,True
3,employment_years,0.651963,False,0.000814,True
4,interest_rate,0.663421,False,-0.000763,True



Interpretation: With a very large sample, even tiny baseline differences can become statistically significant. Therefore, standardized mean differences are more informative for judging whether randomization created practically comparable groups.


## 13. Categorical balance — chi-square tests


In [16]:
categorical_features = [
    col for col in ["grade", "term"]
    if col in df.columns
]

chi_square_results = []

for feature in categorical_features:
    contingency = pd.crosstab(
        df[feature],
        df["policy_group"]
    )

    chi2, p_val, dof, expected = stats.chi2_contingency(
        contingency
    )

    chi_square_results.append({
        "feature": feature,
        "chi2": chi2,
        "degrees_of_freedom": dof,
        "p_value": p_val
    })

chi_square_results = pd.DataFrame(
    chi_square_results
)

display(chi_square_results)


,feature,chi2,degrees_of_freedom,p_value
0,grade,4.030052,6,0.672609
1,term,0.376409,1,0.539532


## 14. Multiple-testing awareness


We have several exploratory balance tests. Running many tests increases the chance of obtaining a small p-value by chance.

For the **primary experiment decision**, we keep the default-rate test as the pre-specified primary outcome. Baseline balance tests are diagnostics rather than additional treatment-effect claims.

This distinction helps avoid turning every exploratory comparison into a headline result.


## 15. Statistical significance vs practical significance


In [17]:
decision_table = pd.DataFrame({
    "Question": [
        "Is default reduction statistically significant?",
        "Is default reduction practically measurable?",
        "Is approval-rate change statistically significant?",
        "Does treatment reduce approvals?",
        "Is the policy economically worthwhile?"
    ],
    "Current Evidence": [
        "See primary Z-test p-value",
        f"{ARR:.2%} absolute reduction",
        "See approval Z-test p-value",
        f"{approval_loss:.2%} approval reduction",
        "Requires Expected Loss analysis"
    ]
})

display(decision_table)

print(
    "\nThe final economic decision is intentionally deferred until "
    "Notebook 06, where PD, LGD, EAD and expected loss are combined "
    "with the approval tradeoff."
)


,Question,Current Evidence
0,Is default reduction statistically significant?,See primary Z-test p-value
1,Is default reduction practically measurable?,1.19% absolute reduction
2,Is approval-rate change statistically signific...,See approval Z-test p-value
3,Does treatment reduce approvals?,2.42% approval reduction
4,Is the policy economically worthwhile?,Requires Expected Loss analysis



The final economic decision is intentionally deferred until Notebook 06, where PD, LGD, EAD and expected loss are combined with the approval tradeoff.


## 16. Executive decision framework


In [18]:
print("=" * 70)
print("CREDIT POLICY — STATISTICAL DECISION SUMMARY")
print("=" * 70)

print(
    f"\nDefault rate — Control:   "
    f"{control_default_rate:.2%}"
)

print(
    f"Default rate — Treatment: "
    f"{treatment_default_rate:.2%}"
)

print(
    f"Absolute default reduction: "
    f"{ARR:.2%}"
)

print(
    f"Relative default reduction: "
    f"{relative_risk_reduction:.2%}"
)

print(
    f"95% CI for Treatment-Control: "
    f"[{ci_low:.2%}, {ci_high:.2%}]"
)

print(
    f"Default Z-statistic: {z_stat:.3f}"
)

print(
    f"Default p-value: {p_value:.6g}"
)

print(
    f"Approval rate — Control:   "
    f"{control_approval_rate:.2%}"
)

print(
    f"Approval rate — Treatment: "
    f"{treatment_approval_rate:.2%}"
)

print(
    f"Approval reduction: "
    f"{approval_loss:.2%}"
)

print(
    f"Estimated NNT: {NNT:.0f}"
)

print("\nDecision status:")

if p_value < alpha and ARR > 0:
    print(
        "✓ Statistical evidence supports a reduction in default."
    )
else:
    print(
        "✗ Statistical evidence does not establish a reduction in default."
    )

if approval_loss > 0:
    print(
        "⚠ The treatment also reduces approvals."
    )

print(
    "→ Final economic recommendation requires Expected Loss analysis."
)

print("=" * 70)


CREDIT POLICY — STATISTICAL DECISION SUMMARY

Default rate — Control:   20.10%
Default rate — Treatment: 18.91%
Absolute default reduction: 1.19%
Relative default reduction: 5.93%
95% CI for Treatment-Control: [-1.33%, -1.06%]
Default Z-statistic: -17.162
Default p-value: 2.54787e-66
Approval rate — Control:   100.00%
Approval rate — Treatment: 97.58%
Approval reduction: 2.42%
Estimated NNT: 84

Decision status:
✓ Statistical evidence supports a reduction in default.
⚠ The treatment also reduces approvals.
→ Final economic recommendation requires Expected Loss analysis.


## 17. Interview-ready interpretation

A strong way to present this experiment is:

> "The simulated policy reduced default by roughly 1.2 percentage points. Because the experiment contains over one million observations, the statistical test has very high power, so I did not rely on the p-value alone. I reported the confidence interval, relative reduction, effect size, and NNT, and then evaluated the approval-rate tradeoff separately. The final policy recommendation is based on expected credit loss rather than statistical significance alone."

### Important limitation

This is a **simulated randomized policy experiment built from historical Lending Club data**. It demonstrates experimental design and statistical analysis, but it is not evidence that Lending Club's real policy would produce the same effect.

### Next notebook

`04_power_analysis.ipynb` will answer:

> **Before running an experiment, how large a sample would we need to reliably detect a 1.2 percentage-point policy effect?**

It will calculate required sample size, statistical power, minimum detectable effect, and sensitivity to different effect sizes.
